# Data Transfer Migrations

Run legacy data migrations against FastQuote database.

**Prerequisites:** `pip install pyodbc`

**Order of execution:**
1. `run_customers` – migrates customers and customer groups
2. `run_contacts` – migrates contacts (depends on customers being migrated first)

In [ ]:
import pyodbc
import os

SERVER = "telquoteweb"
DATABASE = "FastQuote"

conn = pyodbc.connect(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SERVER};DATABASE={DATABASE};Trusted_Connection=yes;",
    autocommit=True,
)

SQL_DIR = os.path.dirname(os.path.abspath("__file__"))


def run_migration(filename: str):
    """Read a .sql file, split on GO batch separators, and execute each batch."""
    filepath = os.path.join(SQL_DIR, filename)
    with open(filepath, "r", encoding="utf-8") as f:
        script = f.read()

    # Remove USE statement — we're already connected to the right database
    batches = script.split("\nGO\n")

    cursor = conn.cursor()
    for i, batch in enumerate(batches, 1):
        batch = batch.strip()
        # Skip empty batches and USE statements
        if not batch or batch.upper().startswith("USE "):
            continue
        try:
            cursor.execute(batch)
            while cursor.description:
                cols = [col[0] for col in cursor.description]
                rows = cursor.fetchall()
                print(f"  Batch {i} result ({', '.join(cols)}):")
                for row in rows:
                    print(f"    {dict(zip(cols, row))}")
                if not cursor.nextset():
                    break
            print(f"  Batch {i}: OK")
        except pyodbc.Error as e:
            print(f"  Batch {i}: ERROR - {e}")
            raise
    cursor.close()
    print("Done.")

## Run Customers Migration

Executes `run_customers_migration.sql`:
- Loads staging tables `_Customers` and `_CustomerGroups` from `oldTelquote`
- Applies city/country overrides from CSV (if configured)
- Populates `dbo.Customers` idempotently
- Maps parent customers and resolves country/city

In [ ]:
def run_customers():
    print("Running customers migration...")
    run_migration("run_customers_migration.sql")

run_customers()

## Run Contacts Migration

Executes `run_contacts_migration.sql`:
- Loads staging table `_Contacts` from `oldTelquote`
- Populates `dbo.Contacts` idempotently
- Maps `Contacts.CustomerID` using `Customers._OldCustomerID`
- Applies `Enabled` / `EmailStatus` logic

**Note:** Customers migration must be run first — contacts depend on `_OldCustomerID` mapping.

In [ ]:
def run_contacts():
    print("Running contacts migration...")
    run_migration("run_contacts_migration.sql")

run_contacts()